In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    precision_recall_curve,
    roc_auc_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_curve,
)



In [ ]:
FIG = Path("../figures")
FIG.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 8



In [ ]:
# ======================
# 1. СИНТЕТИЧЕСКИЙ КРЕДИТНЫЙ ДАТАСЕТ
# ======================
X, y = make_classification(
    n_samples=6000,
    n_features=20,
    n_informative=8,
    weights=[0.9, 0.1],
    random_state=RANDOM_SEED,
)
feature_names = [f"f{i}" for i in range(X.shape[1])]
df = pd.DataFrame(X, columns=feature_names)
df["default"] = y



In [ ]:
print("Number of samples:", df.shape[0])
print("Default rate:", round(y.mean(), 4))
print("Columns:", list(df.columns))



In [ ]:
df.head()



In [ ]:
df.describe().T.head(8)



In [ ]:
# Баланс классов default / non-default
class_counts = df["default"].value_counts().sort_index()
names = ["non-default", "default"]

plt.figure(figsize=(7, 4))
sns.barplot(x=names, y=class_counts.values, hue=names, palette="viridis", legend=False)
plt.title("Default class balance", fontsize=14, pad=12)
plt.ylabel("Count")
plt.grid(axis="y", linestyle="--", alpha=0.7)
for i, c in enumerate(class_counts.values):
    plt.text(i, c + 30, str(c), ha="center")
plt.tight_layout()
plt.show()



In [ ]:
# Распределения ключевых признаков по классам
fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, col in zip(axes.flatten(), feature_names[:6]):
    sns.kdeplot(data=df, x=col, hue="default", ax=ax, palette="viridis", common_norm=False)
    ax.set_title(col)
plt.suptitle("Feature densities by default", fontsize=14)
plt.tight_layout()
plt.show()



In [ ]:
# ======================
# 2. TRAIN / TEST SPLIT
# ======================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.25, random_state=RANDOM_SEED
)
print("Train:", X_train.shape, "Test:", X_test.shape)



In [ ]:
# ======================
# 3. МОДЕЛЬ
# ======================
model = HistGradientBoostingClassifier(random_state=RANDOM_SEED)
model.fit(X_train, y_train)
proba = model.predict_proba(X_test)[:, 1]
print("ROC-AUC:", round(roc_auc_score(y_test, proba), 4))



In [ ]:
# ======================
# 4. ПОДБОР ПОРОГА ПО F1
# ======================
prec, rec, thr = precision_recall_curve(y_test, proba)
f1s = 2 * prec * rec / (prec + rec + 1e-9)
best_i = int(np.argmax(f1s))
best_thr = thr[min(best_i, len(thr) - 1)]
print("best_thr:", round(best_thr, 4), "best_f1:", round(f1s[best_i], 4))
print("f1@0.5:", round(f1_score(y_test, (proba >= 0.5).astype(int)), 4))



In [ ]:
pred_best = (proba >= best_thr).astype(int)
print(classification_report(y_test, pred_best, digits=3, target_names=["non-default", "default"]))



In [ ]:
# PR-кривая и выбранный порог
plt.figure(figsize=(7, 5))
plt.plot(rec, prec, color="#4c72b0", label="PR curve")
plt.scatter(rec[best_i], prec[best_i], color="red", zorder=5, label=f"best F1 @{best_thr:.2f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall curve", fontsize=14, pad=12)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
# ROC-кривая
fpr, tpr, _ = roc_curve(y_test, proba)
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"AUC={roc_auc_score(y_test, proba):.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("FPR")
plt.ylabel("TPR")
plt.title("ROC curve")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()



In [ ]:
# Матрица ошибок при оптимальном пороге
cm = confusion_matrix(y_test, pred_best)
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["non-default", "default"],
    yticklabels=["non-default", "default"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix (thr={best_thr:.2f})")
plt.tight_layout()
plt.show()



In [ ]:
# Важность признаков (permutation-free proxy: абсолютная корреляция с target * model signal)
# Для HGB используем простой proxy через |corr| * scale — и отдельный RF для importance PNG
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train, y_train)
imp = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=True)

plt.figure(figsize=(8, 7))
imp.plot(kind="barh", color=sns.color_palette("viridis", len(imp)))
plt.title("Feature importance (RandomForest proxy)", fontsize=14, pad=12)
plt.xlabel("Importance")
plt.tight_layout()
plt.savefig(FIG / "credit_feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()



In [ ]:
# Сравнение F1 при разных порогах
thresholds = np.linspace(0.1, 0.9, 17)
f1_thr = [f1_score(y_test, (proba >= t).astype(int)) for t in thresholds]

plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_thr, marker="o")
plt.axvline(best_thr, color="red", linestyle="--", label=f"best={best_thr:.2f}")
plt.axvline(0.5, color="gray", linestyle=":", label="0.5")
plt.xlabel("Threshold")
plt.ylabel("F1")
plt.title("F1 vs decision threshold")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

